In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/upworthy-archive-exploratory-packages-03.12.2020.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst rows: \n", df.head())
print("\nData types: \n", df.info())
print("\nMissing values: \n", df.isnull().sum())


Shape: (22666, 17)

Columns: ['Unnamed: 0', 'created_at', 'updated_at', 'clickability_test_id', 'excerpt', 'headline', 'lede', 'slug', 'eyecatcher_id', 'impressions', 'clicks', 'significance', 'first_place', 'winner', 'share_text', 'square', 'test_week']

First rows: 
    Unnamed: 0               created_at               updated_at  \
0           0  2014-11-20 06:43:16.005  2016-04-02 16:33:38.062   
1           1  2014-11-20 06:43:44.646  2016-04-02 16:25:54.021   
2           2  2014-11-20 06:44:59.804  2016-04-02 16:25:54.024   
3           3  2014-11-20 06:54:36.335  2016-04-02 16:25:54.027   
4           4  2014-11-20 06:54:57.878  2016-04-02 16:31:45.671   

       clickability_test_id                           excerpt  \
0  546d88fb84ad38b2ce000024  Things that matter. Pass 'em on.   
1  546d88fb84ad38b2ce000024  Things that matter. Pass 'em on.   
2  546d88fb84ad38b2ce000024  Things that matter. Pass 'em on.   
3  546d902c26714c6c44000039  Things that matter. Pass 'em on.   
4 

In [7]:
#Compute CTR and an overview of experiment structure

df['created_at'] = pd.to_datetime(df['created_at'], format='mixed', errors='coerce')


# CTR = click-through rate = clicks/impressions

df['ctr'] = df['clicks'] / df['impressions']


#Experiments structure

experiments_sizes = df.groupby('clickability_test_id').size()

print("=== Experiment Structure===")
print(f"Total packages (headline variants): {len(df):,}")
print(f"Total unique experiments: {df['clickability_test_id'].nunique():,}")

print(f"\nVariants per experiment: {experiments_sizes.describe().round(2)}")
print(f"\nDistribution of variants per experiment: \n{experiments_sizes.value_counts().sort_index().head(10)}")

#CTR distribution
print(f"\n=== CTR Distribution ===")
print(f"Mean CTR:   {df['ctr'].mean():.4f} ({df['ctr'].mean()*100:.2f}%)")
print(f"Median CTR: {df['ctr'].median():.4f} ({df['ctr'].median()*100:.2f}%)")
print(f"Std CTR:    {df['ctr'].std():.4f}")
print(f"Min CTR:    {df['ctr'].min():.4f}")
print(f"Max CTR:    {df['ctr'].max():.4f}")


#Impressions distribution
print(f"\n=== Impressions Distribution ===")
print(f"Mean impressions per package:   {df['impressions'].mean():,.0f}")
print(f"Median impressions per package: {df['impressions'].median():,.0f}")
print(f"Min: {df['impressions'].min():,}  Max: {df['impressions'].max():,}")

#Filter bad randomization period
# because researcher found sample ratio mismatch, we've to remove data b/w june 25, 2013 - jan 1, 2014
df_clean = df[~((df['created_at'] >= '2013-06-25') & (df['created_at'] <= '2014-01-10'))]

print(f"\n=== Data Quality Filter ===")
print(f"Rows before removing bad randomization period: {len(df):,}")
print(f"Rows after: {len(df_clean):,}")
print(f"Rows removed: {len(df) - len(df_clean):,}")


#save cleaned data for next steps
df_clean.to_csv('../data/processed/upworthy_clean.csv', index=False)
print(f"\nClean data saved to data/processed/upworthy_clean.csv")




=== Experiment Structure===
Total packages (headline variants): 22,666
Total unique experiments: 4,873

Variants per experiment: count    4873.00
mean        4.65
std         1.19
min         2.00
25%         4.00
50%         4.00
75%         5.00
max        14.00
dtype: float64

Distribution of variants per experiment: 
2      127
3      182
4     2516
5      928
6      865
7      134
8       86
9       18
10       7
11       8
Name: count, dtype: int64

=== CTR Distribution ===
Mean CTR:   0.0158 (1.58%)
Median CTR: 0.0125 (1.25%)
Std CTR:    0.0123
Min CTR:    0.0000
Max CTR:    0.1361

=== Impressions Distribution ===
Mean impressions per package:   3,575
Median impressions per package: 3,122
Min: 13  Max: 25,314

=== Data Quality Filter ===
Rows before removing bad randomization period: 22,666
Rows after: 18,415
Rows removed: 4,251

Clean data saved to data/processed/upworthy_clean.csv


In [ ]:
# Exploring our dataset and its distributions
# 
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = df_clean.copy()
# checking for infinite or zero ctr, which can break all stat or cause division errors
print("===CTR data===")
print(f"Infinite CTR Values: {np.isinf(df['ctr']).sum()}")
print(f"Nan ctr values: {df['ctr'].isnull().sum()}")
print(f'Zero ctr values: {(df['ctr'] == 0).sum()}')
print(f'CTR values > 1: {(df['ctr'] > 1).sum()}')

#pratcically, filter packages with les than 100 impressions
thresholds = [10, 50, 100, 200, 500]
for t in thresholds:
    remaining = (df['impressions'] >= t).sum()
    pct = remaining / len(df) * 100
    print(f"  >= {t:4d} impressions: {remaining:,} packages ({pct:.1f}% retained)")
#100 threshold is the standard in experimentation
df_filtered = df[df['impressions'] >= 100].copy()
print(f"\n After filtering for >= 100 impressions: {len(df_filtered):,} of {len(df):,}")

# Now we are going to do experiment-level analysis, so we need to aggregate data by clickability_test_id
# and for each we copute the winning nad losing ctr

exp_stats = df_filtered.groupby('clickability_test_id').agg(
    n_variants = ('ctr', 'count'),
    mean_ctr = ('ctr', 'mean'),
    max_ctr = ('ctr', 'max'),
    min_ctr = ('ctr', 'min'),
    total_impr = ('impressions', 'sum'),
    total_clicks = ('clicks', 'sum')).reset_index()

# Compute the effect size per experiment. How much better was the best headline vs. the wost?
exp_stats['ctr_range'] = exp_stats['max_ctr'] - exp_stats['min_ctr']
exp_stats['rel_lift'] = (exp_stats['ctr_range']) / exp_stats['mean_ctr']


# Now we can look at the distribution of effect sizes across experiments
print("\n=== Experiment Effect Sizes ===")
print(f" Mean CTR range: {exp_stats['ctr_range'].mean():.4f} ({exp_stats['ctr_range'].mean()*100:.2f}%)")
print(f" Median CTR range: {exp_stats['ctr_range'].median():.4f}")
print(f" Mean relative lift: {exp_stats['rel_lift'].mean():.4f}")
print(f" Median relative lift: {exp_stats['rel_lift'].median():.4f}")

#Distribution check
# Is CTR normally distributed?
# skewness > 0 = right skewed, skewness < 0 = left skewed
# kurtosis > 0 = heavier tails than normal
print("\n===Distribution shape of CTR===")
print(f"Skewness: {stats.skew(df_filtered['ctr']):.3f}")
print(f"Kurtosis: {stats.kurtosis(df_filtered['ctr']):.3f}")
print(f"(Skewness and kurtosis near 0 means normally distributed)")

# Shapiro-Wilk normality test
# H0: data is normally distributed
# p < 0.05 → reject H0 → not normal
# We sample 500 rows — Shapiro-Wilk gets slow on large datasets
sample = df_filtered['ctr'].sample(500, random_state = 42)
_, p_normal = stats.shapiro(sample)
print(f"Shapiro-Wilk p-value: {p_normal:.6f}")
print(f"Normal? {'Yes' if p_normal > 0.05 else 'No - which is expted for ctr data'}")


#saving eperiment level stats
exp_stats.to_csv('../data/processed/experiment_stats.csv', index = False)
print(f"\nExperiment stats saved")
print(f"\n First rows: \n {exp_stats.head()}")